# 🐍 Instagram Geri Takip Etmeyenler Analiz Uygulaması

Bu Jupyter Notebook çalışmasında, Instagram hesabınızın takipçi (`followers.html`) ve takip edilenler (`following.html`) listelerini karşılaştırarak **sizi geri takip etmeyen kişileri** (unfollowers) tespit eden, sonuçları Excel veya PDF olarak raporlayabilen ve bu kişileri tarayıcıda otomatik açabilen gelişmiş bir masaüstü yazılımının **adım adım tasarımını, kod yapısını ve algoritmalarını** açıklayacağız.

Bu notebook, projeyi inceleyen kişinin hem **kullanılan kütüphaneleri sıfırdan öğrenmesini**, hem **arkadaki matematiksel algoritmayı kavramasını**, hem de **kodun satır satır mantığını çözmesini** amaçlar. Son derece detaylı açıklamalar ve görsel/matematiksel modellerle hazırlanmıştır.

### 🧠 Neler Öğreneceksiniz?
1. **Python Kütüphaneleri**: `pandas`, `BeautifulSoup` (`bs4`), `fpdf`, `tkinter`, `threading` ve `webbrowser` modüllerinin kullanım amaçları.
2. **HTML Ayrıştırma**: Instagram verilerinden profil isimlerini ve linklerini `BeautifulSoup` ile süzme.
3. **Küme Farkı Algoritması**: Bizi geri takip etmeyen kişileri matematiksel küme mantığıyla Python'da bulma.
4. **Çoklu İş Parçacığı (Multi-threading)**: Dosyalar okunurken arayüzün kilitlenmesini veya donmasını engellemek için `threading.Thread` kullanımı.
5. **Dinamik Arama Filtresi (Trace)**: Arama çubuğuna harf girildiği anda (Key Release beklemeden) listenin anlık olarak süzülmesi.
6. **Dosya Raporlama**: Pandas ile Excel (.xlsx) ve FPDF ile tıklanabilir linklere sahip PDF formatında rapor oluşturma.
7. **Tkinter Arayüzü**: Koyu (hacker) tema tasarımı, Treeview tablosu ve Scrollbar entegrasyonu.

---
**NOT:** Hücreleri sırayla çalıştırmak için hücreyi seçip **Shift + Enter** tuş kombinasyonunu kullanabilirsiniz.

## 1. Bölüm: Kullanılan Kütüphaneler ve Seçilme Nedenleri

Projede kullanılan kütüphanelerin detaylı analizleri:

* **`pandas`**:
  * **Nedir?** Python'un en popüler veri analizi ve manipülasyonu kütüphanesidir.
  * **Amacı**: Çekilen takipçi listelerini tek boyutlu bir seri veya tabloya (DataFrame) dönüştürerek kolayca Excel (`.xlsx`) dosyası olarak kaydetmemizi sağlar. Excel kaydı için arka planda `openpyxl` motorunu kullanır.

* **`BeautifulSoup` (`bs4` kütüphanesinden)**:
  * **Nedir?** HTML ve XML belgelerini kolayca sorgulamamızı sağlayan popüler bir kütüphanedir.
  * **Amacı**: Instagram'dan indirilen `followers.html` ve `following.html` dosyalarının içindeki karmaşık HTML etiketlerini parse ederek, sadece kullanıcı adlarını taşıyan link etiketlerini ayıklar.

* **`fpdf`**:
  * **Nedir?** Python ile dinamik PDF belgeleri oluşturmak için kullanılan hafif bir kütüphanedir.
  * **Amacı**: Geri takip etmeyenlerin listesini, tıklanabilir Instagram profil linkleriyle birlikte düzenli bir PDF raporuna dönüştürür.

* **`threading`**:
  * **Nedir?** Python'un standart kütüphanesinde yer alan iş parçacığı (thread) yönetim modülüdür.
  * **Amacı**: Dosyaların okunması ve HTML ayrıştırma gibi süre alan işlemleri arka planda çalıştırarak Tkinter arayüzünün donmasını, yanıt vermiyor durumuna düşmesini engeller.

* **`webbrowser`**:
  * **Nedir?** Sistemde varsayılan olarak ayarlı olan internet tarayıcısını açıp belirli bir adrese yönlendirmeyi sağlayan standart modüldür.
  * **Amacı**: Listede tespit edilen kişilerin Instagram profillerini tek tıkla tarayıcıda yeni sekmeler olarak açmamızı sağlar.

In [ ]:
# Gerekli kütüphaneleri içe aktaralım
import tkinter as tk
from tkinter import ttk, filedialog, messagebox, StringVar
import os
from bs4 import BeautifulSoup
from fpdf import FPDF
import threading
import webbrowser
import pandas as pd

## 2. Bölüm: HTML Ayrıştırma ve Instagram Veri Yapısı

Instagram ayarlarınızdan indirdiğiniz takip verileri HTML formatındadır. Bu dosyalar, tarayıcıda açıldığında birer tablo veya liste gibi görünse de arka planda hiyerarşik HTML etiketleri içerir.

### 2.1 Instagram HTML Yapısı
Instagram'ın profil sayfalarına ait bağlantıları ve kullanıcı adlarını barındıran etiketler şu şekildedir:
```html
<a target="_blank" href="https://www.instagram.com/alican.kaya">alican.kaya</a>
```

Bu yapıyı BeautifulSoup ile ayrıştırırken:
1. Dosya içeriğini utf-8 kodlamasıyla açar ve metin olarak okuruz.
2. `BeautifulSoup(html_content, "html.parser")` ile parse ağacını kurarız.
3. `soup.find_all("a", target="_blank")` metodunu kullanarak hedefi yeni sekmede açmak üzere ayarlanmış (`target="_blank"`) olan tüm link etiketlerini buluruz.
4. Bu etiketlerin içerisindeki saf metni (`item.get_text()`) alarak Instagram kullanıcı adını elde ederiz.

### 2.2 Çalışma Dizini Bağımsızlığı (os.path)
Uygulama çalıştırıldığında dosyaların aranacağı klasör, kullanıcının komut satırında bulunduğu aktif dizine (CWD) göre değişiklik gösterebilir. Eğer `followers_1.html` dosyasını doğrudan `"followers_1.html"` şeklinde ararsak, programı başka bir klasörden başlattığımızda dosya bulunamadı hatası alırız.

Bunu önlemek için python scriptinin bulunduğu tam klasör yolunu dinamik olarak tespit ederiz:
```python
script_dir = os.path.dirname(os.path.abspath(__file__))
file_path = os.path.join(script_dir, "followers_1.html")
```
Bu sayede program hangi klasörden çalıştırılırsa çalıştırılsın, kendi klasöründeki dosyaları hatasız bir şekilde bulur.

In [ ]:
# Script dizinini tespit etme ve dosya yolları testi
script_dir = os.path.dirname(os.path.abspath("__file__")) # Jupyter içinde "__file__" yerine "__file__" dizesi alınmıştır
print("Proje Dizin Yolu:", script_dir)

# followers_1.html'den followers_99.html'ye kadar olan dosyaları otomatik arama testi
dosyalar = []
for i in range(1, 100):
    yol = os.path.join(script_dir, f"followers_{i}.html")
    if os.path.exists(yol):
        dosyalar.append(yol)
    else:
        break

print(f"Otomatik Tespit Edilen Takipçi Dosyası Sayısı: {len(dosyalar)}")

## 3. Bölüm: Küme Farkı Algoritması ve Karşılaştırma Mantığı

Uygulamanın ana amacı sizi geri takip etmeyen kişileri bulmaktır. Matematiksel olarak bu işlem **Küme Farkı** (Set Difference) işlemidir.

### 3.1 Küme Teorisi
* **A Kümesi (Takip Edilenler - Following)**: Sizin Instagram'da takip ettiğiniz tüm kişilerin listesidir.
* **B Kümesi (Takipçiler - Followers)**: Sizi takip eden tüm kişilerin listesidir.
* **Fark Kümesi ($A \setminus B$)**: Sizin takip ettiğiniz ama sizi takip etmeyen kişilerdir.

Python'da bu farkı listeler üzerinden list comprehension (liste üreteci) ile şu şekilde buluruz:
```python
non_followers_list = [follower for follower in following_list if follower not in followers_list]
```
Bu kod satırı, `following_list` (takip edilenler) içerisindeki her bir kişiyi tek tek kontrol eder ve eğer bu kişi `followers_list` (takipçiler) içinde **yer almıyorsa**, onu `non_followers_list` içerisine dahil eder.

In [ ]:
# Küçük bir örnekle küme farkı algoritmasını test edelim
takip_edilenler = ["alican", "tuncay", "veli", "ayse", "mehmet", "fatma"]
takipci_listesi = ["alican", "veli", "fatma"]

# Geri takip etmeyenleri bulalım: takip_edilenler içinde olup takipci_listesi içinde olmayanlar
geri_takip_etmeyenler = [kisi for kisi in takip_edilenler if kisi not in takipci_listesi]

print("Takip Edilenler (Following):", takip_edilenler)
print("Takip Edenler (Followers):  ", takipci_listesi)
print("--------------------------------------------------")
print("Geri Takip Etmeyenler (Unfollowers):", geri_takip_etmeyenler)

## 4. Bölüm: Çoklu İş Parçacığı (Multi-threading) ile Akıcı Arayüz

Grafik arayüz (GUI) kütüphaneleri (Tkinter gibi) tek bir ana iş parçacığı (**Main Thread**) üzerinde çalışır. Bu iş parçacığı ekranı çizer, tıklamaları yakalar ve buton komutlarını çalıştırır.

### 4.1 UI Donması Neden Olur?
Eğer ana iş parçacığı üzerinde büyük bir dosyayı okumaya başlarsanız (örneğin 10.000 satırlık HTML ayrıştırma), Python bu dosya okuma işlemi bitene kadar arayüzü güncelleyemez. Kullanıcı ekrana tıkladığında işletim sistemi *"Program Yanıt Vermiyor"* uyarısı verir ve ekran beyazlaşır/donar.

### 4.2 Çözüm: Thread Kullanımı
Bunu engellemek için, dosya okuma ve analiz yapma gibi ağır işleri **ayrı bir iş parçacığı (Thread)** içine göndeririz.
```python
t = threading.Thread(target=find_non_followers)
t.start()
```
Bu sayede:
* **Yeni Thread**: Arka planda `following.html` ve `followers.html` dosyalarını okur, BeautifulSoup ile ayrıştırır ve karşılaştırma yapar.
* **Main Thread (Arayüz)**: Kullanıcıya donmadan çalışmaya devam eder, animasyonları çizer veya "Hesaplanıyor..." yazısını gösterir.
* Arka plandaki iş bittiğinde `update_table` çağrısıyla sonuçlar tabloya basılır.

## 5. Bölüm: Dinamik Arama Filtresi (StringVar Trace)

Geleneksel arayüzlerde kullanıcı arama kutusuna yazar, ardından "Ara" butonuna basar. Bu durum kullanıcı deneyimini yavaşlatır.

Modern uygulamalarda arama işlemi **kullanıcı klavyede tuşa bastığı anda** gerçekleşir. Tkinter'da bunu yapabilmek için **`StringVar`** nesnesinin **`trace_add`** metodunu kullanırız.

```python
search_var = tk.StringVar()
search_var.trace_add("write", lambda *args: search_followers())
```

* `search_var`: Arama kutusuna (`Entry`) bağlanan dinamik bir string değişkenidir.
* `trace_add("write", ...)`: Bu değişkenin değeri her değiştiğinde (kullanıcı her yazı yazdığında veya sildiğinde) belirtilen fonksiyonu (`search_followers`) otomatik olarak tetikler.
* Bu sayede kullanıcı harf girdiği milisaniyede tablo dinamik olarak güncellenir ve filtreleme anlık hissettirir.

## 6. Bölüm: Sonuçları Raporlama ve Dışa Aktarma

Analiz bittikten sonra sonuçları arşivlemek için iki popüler formatı (Excel ve PDF) destekliyoruz.

### 6.1 Excel Formatı (Pandas & Openpyxl)
Pandas kütüphanesi verileri sütun bazlı saklar. Bir Python listesini Excel'e aktarmak oldukça basittir:
```python
df = pd.DataFrame({"Kullanıcı Adı": active_list})
df.to_excel(excel_file_path, index=False)
```
Bu kod, listemizi sütun adı `"Kullanıcı Adı"` olan bir tabloya çevirir ve `to_excel` metoduyla diske kaydeder.

### 6.2 PDF Raporlama (FPDF & Profil Bağlantıları)
FPDF kütüphanesi ile koordinat bazlı PDF sayfaları tasarlayabiliriz. Programımız geri takip etmeyen her bir kullanıcının kullanıcı adını yazarken, hemen altına tıklanabilir bir web linki ekler:
```python
pdf.cell(200, 8, txt=f"- {follower_username}", ln=True)
pdf.cell(200, 8, txt=f"  Profil Linki: https://www.instagram.com/{follower_username}/", ln=True)
```
Bu sayede kullanıcı PDF raporunu açtığında direkt olarak linke tıklayarak tarayıcıda o profile gidebilir.

In [ ]:
# Arayüz olmadan veri okuma ve karşılaştırma fonksiyonlarının simüle edilmesi
def simulation_run():
    # followers_1.html ve following.html dosyalarınız script klasöründeyse test edebilirsiniz.
    # Aksi takdirde dosya bulunamadı yazacaktır.
    script_dir = os.path.dirname(os.path.abspath("__file__"))
    following_path = os.path.join(script_dir, "following.html")
    
    print("Simülasyon başlatıldı...")
    if not os.path.exists(following_path):
        print("Simülasyon Uyarısı: 'following.html' test dosyası bulunamadı. Lütfen analiz için dosyalarınızı klasöre ekleyin.")
        return
        
    try:
        with open(following_path, "r", encoding="utf-8") as f:
            soup = BeautifulSoup(f.read(), "html.parser")
        following = [item.get_text() for item in soup.find_all("a", target="_blank")]
        print(f"Toplam Takip Edilen (Following) Sayısı: {len(following)}")
    except Exception as e:
        print(f"Hata: {e}")

simulation_run()

## 7. Bölüm: Grafiksel Kullanıcı Arayüzü (Tkinter GUI) ve Tam Kod

Aşağıdaki hücrede, projenin tüm mantıksal katmanlarını ve Tkinter GUI kodunu bir araya getiren **tam çalıştırılabilir kod bloğu** yer almaktadır.

> ⚠️ **ÖNEMLİ UYARI:** Aşağıdaki kodu çalıştırdığınızda bilgisayarınızda görsel bir masaüstü penceresi açılacaktır. Uygulamayı pencere köşesindeki **[X]** (kapat) butonuyla kapatana kadar bu hücre meşgul durumda kalacaktır. Hücre çalışırken solundaki `[*]` işareti meşgul olduğunu gösterir. Pencereyi kapattığınızda hücrenin çalışması tamamlanacaktır.

In [ ]:
import tkinter as tk
from tkinter import ttk, filedialog, messagebox, StringVar
import os
from bs4 import BeautifulSoup
from fpdf import FPDF
import threading
import webbrowser
import pandas as pd

# Global Değişkenler
followers_list = []
active_list = []
file_paths = []

def update_table(data):
    table.delete(*table.get_children())
    for index, item in enumerate(data, start=1):
        table.insert("", "end", values=(index, item))

def open_files_and_show_followers():
    global file_paths
    file_paths = []
    script_dir = os.path.dirname(os.path.abspath(__file__))
    for i in range(1, 100):
        file_path = os.path.join(script_dir, f"followers_{i}.html")
        if os.path.exists(file_path):
            file_paths.append(file_path)
        else:
            break
    
    if not file_paths:
        messagebox.showwarning("Uyarı", "followers_X.html dosyası bulunamadı!")
        return
    
    file_listbox.delete(0, tk.END)
    for file_path in file_paths:
        file_listbox.insert(tk.END, os.path.basename(file_path))
    show_all_followers()

def open_file_and_show_followers():
    global file_paths
    file_paths = filedialog.askopenfilenames(title="Dosya Seç", filetypes=[("HTML Files", "*.html")])
    if not file_paths:
        return
    file_listbox.delete(0, tk.END)
    for file_path in file_paths:
        file_listbox.insert(tk.END, os.path.basename(file_path))
    show_all_followers()

def show_all_followers():
    if not file_paths:
        messagebox.showwarning("Uyarı", "Lütfen önce dosyaları seçin veya otomatik çekmeyi deneyin.")
        return
    t = threading.Thread(target=show_followers, args=(file_paths,))
    t.start()

def show_followers(file_paths):
    global followers_list, active_list
    followers_list.clear()
    progress_label.config(text="Takipçiler yükleniyor...")
    for file_path in file_paths:
        try:
            with open(file_path, "r", encoding="utf-8") as file:
                html_content = file.read()
            soup = BeautifulSoup(html_content, "html.parser")
            names = [item.get_text() for item in soup.find_all("a", target="_blank")]
            followers_list.extend(names)
        except Exception as e:
            messagebox.showerror("Hata", f"{os.path.basename(file_path)} dosyası okunurken hata oluştu:\n{e}")
            progress_label.config(text="Hata oluştu.")
            return

    active_list = list(followers_list)
    update_table(active_list)
    progress_label.config(text="Takipçiler başarıyla yüklendi.")
    show_followers_count()

def show_followers_count():
    if followers_list:
        messagebox.showinfo("Toplam Takipçi Sayısı", f"Toplam Takipçi Sayısı: {len(followers_list)}")
    else:
        messagebox.showwarning("Uyarı", "Henüz takipçi bulunmamaktadır.")

def show_non_followers():
    if not file_paths:
        messagebox.showwarning("Uyarı", "Lütfen önce takipçi dosyalarını seçin.")
        return
    t = threading.Thread(target=find_non_followers)
    t.start()

def find_non_followers():
    global followers_list, active_list
    progress_label.config(text="Takip etmeyenler bulunuyor...")
    script_dir = os.path.dirname(os.path.abspath(__file__))
    following_path = os.path.join(script_dir, "following.html")
    
    if not os.path.exists(following_path):
        following_path = "following.html"
        if not os.path.exists(following_path):
            messagebox.showerror("Hata", "following.html dosyası bulunamadı!\nLütfen dosyayı script klasörüne ekleyin.")
            progress_label.config(text="Hata: following.html bulunamadı.")
            return

    try:
        with open(following_path, "r", encoding="utf-8") as file:
            html_content = file.read()
        soup = BeautifulSoup(html_content, "html.parser")
        following_list = [item.get_text() for item in soup.find_all("a", target="_blank")]
    except Exception as e:
        messagebox.showerror("Hata", f"following.html dosyası okunurken hata oluştu:\n{e}")
        progress_label.config(text="Hata oluştu.")
        return

    followers_list.clear()
    for file_path in file_paths:
        try:
            with open(file_path, "r", encoding="utf-8") as file:
                html_content = file.read()
            soup = BeautifulSoup(html_content, "html.parser")
            names = [item.get_text() for item in soup.find_all("a", target="_blank")]
            followers_list.extend(names)
        except Exception:
            continue

    non_followers_list = [follower for follower in following_list if follower not in followers_list]
    active_list = list(non_followers_list)
    update_table(active_list)
    progress_label.config(text="Geri takip etmeyenler başarıyla listelendi.")

def search_followers():
    search_text = search_var.get().lower()
    if search_text:
        search_result = [item for item in active_list if search_text in item.lower()]
        update_table(search_result)
    else:
        update_table(active_list)

def clear_search():
    search_var.set("")
    update_table(active_list)

def export_to_excel():
    if active_list:
        df = pd.DataFrame({"Kullanıcı Adı": active_list})
        excel_file_path = filedialog.asksaveasfilename(defaultextension=".xlsx", filetypes=[("Excel files", "*.xlsx")])
        if excel_file_path:
            try:
                df.to_excel(excel_file_path, index=False)
                messagebox.showinfo("Başarılı", "Excel dosyası başarıyla oluşturuldu!")
            except Exception as e:
                messagebox.showerror("Hata", f"Excel kaydedilirken hata oluştu:\n{e}")
    else:
        messagebox.showwarning("Uyarı", "Aktarılacak herhangi bir veri bulunmamaktadır.")

def export_non_followers_to_pdf():
    table_rows = table.get_children()
    if table_rows:
        pdf = FPDF()
        pdf.add_page()
        pdf.set_font("Arial", size=12)
        pdf.cell(200, 10, txt="Instagram Geri Takip Etmeyenler Listesi", ln=True, align='C')
        pdf.ln(10)

        for row in table_rows:
            follower_username = table.item(row)["values"][1]
            pdf.cell(200, 8, txt=f"- {follower_username}", ln=True)
            pdf.cell(200, 8, txt=f"  Profil Linki: https://www.instagram.com/{follower_username}/", ln=True)
            pdf.ln(2)

        pdf_file_path = filedialog.asksaveasfilename(defaultextension=".pdf", filetypes=[("PDF files", "*.pdf")])
        if pdf_file_path:
            try:
                pdf.output(pdf_file_path)
                messagebox.showinfo("Başarılı", "PDF dosyası başarıyla oluşturuldu!")
            except Exception as e:
                messagebox.showerror("Hata", f"PDF kaydedilirken hata oluştu:\n{e}")
    else:
        messagebox.showwarning("Uyarı", "Aktarılacak herhangi bir kişi bulunmamaktadır.")

def open_non_followers_in_browser():
    table_rows = table.get_children()
    if not table_rows:
        messagebox.showwarning("Uyarı", "Tarayıcıda açılacak herhangi bir kullanıcı bulunmamaktadır.")
        return
    for row in table_rows:
        follower_username = table.item(row)["values"][1]
        webbrowser.open(f"https://www.instagram.com/{follower_username}/")

# --- Tkinter Arayüz Kurulumu ---
app = tk.Tk()
app.title("Takipçi Analiz Uygulaması V2 @Tuncay.Lore")
app.geometry("1000x700")
app.configure(bg="#0f0f0f")

title_label = tk.Label(app, text="Takipçi Karşılaştırma Uygulaması", font=("Courier", 24, "bold"), bg="#0f0f0f", fg="#4caf50", pady=20)
title_label.pack(fill="x")

button_frame = tk.Frame(app, bg="#0f0f0f")
button_frame.pack(pady=20)

select_file_button = tk.Button(button_frame, text="Dosyaları Seç ve Takipçileri Göster", command=open_file_and_show_followers, bg="#4caf50", fg="white", padx=10, pady=5, bd=0)
select_file_button.grid(row=0, column=0, padx=10)

select_auto_fetch_button = tk.Button(button_frame, text="Otomatik Çek", command=open_files_and_show_followers, bg="#4caf50", fg="white", padx=10, pady=5, bd=0)
select_auto_fetch_button.grid(row=0, column=1, padx=10)

show_all_button = tk.Button(button_frame, text="Tüm Takipçileri Göster", command=show_all_followers, bg="#4caf50", fg="white", padx=10, pady=5, bd=0)
show_all_button.grid(row=0, column=2, padx=10)

show_non_followers_button = tk.Button(button_frame, text="Takip Etmeyenleri Göster", command=show_non_followers, bg="#4caf50", fg="white", padx=10, pady=5, bd=0)
show_non_followers_button.grid(row=0, column=3, padx=10)

export_to_excel_button = tk.Button(button_frame, text="Excel'e Aktar", command=export_to_excel, bg="#4caf50", fg="white", padx=10, pady=5, bd=0)
export_to_excel_button.grid(row=0, column=4, padx=10)

export_to_pdf_button = tk.Button(button_frame, text="PDF'e Aktar", command=export_non_followers_to_pdf, bg="#4caf50", fg="white", padx=10, pady=5, bd=0)
export_to_pdf_button.grid(row=0, column=5, padx=10)

open_in_browser_button = tk.Button(button_frame, text="Tarayıcıda Aç", command=open_non_followers_in_browser, bg="#4caf50", fg="white", padx=10, pady=5, bd=0)
open_in_browser_button.grid(row=0, column=6, padx=10)

progress_label = tk.Label(app, text="", bg="#0f0f0f", fg="#4caf50")
progress_label.pack(pady=5)

file_listbox = tk.Listbox(app, width=90, height=4, bg="#212121", fg="white", borderwidth=0, highlightthickness=0)
file_listbox.pack(pady=10)

search_var = tk.StringVar()
search_var.trace_add("write", lambda *args: search_followers())

search_entry = ttk.Entry(app, textvariable=search_var, width=70, font=("Courier", 12), style="search.TEntry")
search_entry.pack(pady=5)

clear_button = tk.Button(app, text="Aramayı Temizle", command=clear_search, bg="#4caf50", fg="#0f0f0f", bd=0, padx=10)
clear_button.pack(pady=5)

frame = tk.Frame(app, bg="#0f0f0f")
frame.pack(padx=10, pady=10)

columns = ("Sıra No.", "İsim")
table = ttk.Treeview(frame, columns=columns, show="headings", style="Custom.Treeview")
table.heading("Sıra No.", text="Sıra No.")
table.heading("İsim", text="İsim")
table.column("Sıra No.", width=100, anchor="center")
table.column("İsim", width=400, anchor="w")
table.pack(side="left")

vsb = ttk.Scrollbar(frame, orient="vertical", command=table.yview)
vsb.pack(side="right", fill="y")
table.configure(yscrollcommand=vsb.set)

style = ttk.Style()
style.configure("search.TEntry", foreground="#4caf50", background="#212121", bordercolor="#4caf50")
style.configure("Custom.Treeview", background="#212121", foreground="white", fieldbackground="#212121", highlightthickness=0)

app.mainloop()